# Diseño de experimentos y control estadístico con Python

Cuaderno del curso. Cada celda lleva el mismo número que en el video (por ejemplo, **Celda 2.4**).

Para ejecutar una celda: clic en ella y Mayúsculas + Enter. Los datos son simulados para enseñar.

## Sección 0 · Bienvenida y configuración

**Celda 0.1** · Instala las librerías del curso (solo en Colab)

In [ ]:
!pip install pyDOE3==1.6.2 statsmodels==0.15.0 -q

**Celda 0.2** · Carga las librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
URL = "https://raw.githubusercontent.com/lara-montano/doe-control-estadistico/main/datos/"
print("Listo")

**Celda 0.3** · Lee la tabla de lotes

In [ ]:
lotes = pd.read_csv(URL + "lotes_fase1.csv")
lotes.head()

**Celda 0.4** · ¿Cuántas filas y columnas?

In [ ]:
lotes.shape

**Celda 0.5** · Una columna: el contenido de ésteres

In [ ]:
esteres = lotes["esteres"]
esteres.head()

**Celda 0.6** · Tu primera gráfica

In [ ]:
plt.plot(lotes["lote"], esteres, marker="o")
plt.xlabel("lote")
plt.ylabel("ésteres (% m/m)")
plt.show()

## Sección 1 · La variación de un proceso

**Celda 1.1** · La media, a mano

In [ ]:
n = len(esteres)
suma = esteres.sum()
media = suma / n
print("n =", n)
print("suma =", round(suma, 2))
print("media =", round(media, 3))

**Celda 1.2** · La desviación estándar, a mano

In [ ]:
desvios = esteres - media
cuadrados = desvios ** 2
varianza = cuadrados.sum() / (n - 1)
de = varianza ** 0.5
print("varianza =", round(varianza, 4))
print("desviación estándar =", round(de, 3))

**Celda 1.3** · Comprueba con pandas

In [ ]:
print("media con pandas:", round(esteres.mean(), 3))
print("desviación con pandas:", round(esteres.std(), 3))

**Celda 1.4** · El histograma y la especificación

In [ ]:
LEI = 96.5   # límite inferior de especificación (EN 14214)
plt.hist(esteres, bins=10, edgecolor="white")
plt.axvline(LEI, color="red", linestyle="--")
plt.xlabel("ésteres (% m/m)")
plt.show()

**Celda 1.5** · ¿Cuántos lotes quedan bajo la especificación?

In [ ]:
bajo = esteres < LEI
print("lotes bajo 96.5 %:", bajo.sum())
print("proporción:", bajo.mean())

**Celda 1.6** · La gráfica de corrida

In [ ]:
mediana = esteres.median()
plt.plot(lotes["lote"], esteres, marker="o")
plt.axhline(mediana, color="teal")
plt.axhline(LEI, color="red", linestyle="--")
plt.show()
print("mediana =", mediana)

**Celda 1.7** · Práctica: la media por proveedor

In [ ]:
por_proveedor = lotes.groupby("proveedor")["esteres"]
print(por_proveedor.mean().round(3))
print(por_proveedor.size())

## Sección 2 · Cartas de control para variables

**Celda 2.1** · Carta I-MR: el rango móvil

In [ ]:
x = esteres.iloc[:40]     # los primeros 40 lotes, uno por uno
rm = x.diff().abs()       # |lote − lote anterior|
print(rm.head(4).round(2).tolist())

**Celda 2.2** · Carta I-MR: la media y el rango móvil promedio

In [ ]:
media_x = x.mean()
rm_prom = rm.mean()
print("media =", round(media_x, 3))
print("rango móvil promedio =", round(rm_prom, 3))

**Celda 2.3** · Carta I-MR: los límites

In [ ]:
lsc_i = media_x + 2.66 * rm_prom
lic_i = media_x - 2.66 * rm_prom
lsc_rm = 3.267 * rm_prom
print("I:  LIC =", round(lic_i, 3), "  LSC =", round(lsc_i, 3))
print("RM: LSC =", round(lsc_rm, 3))

**Celda 2.4** · Carta I-MR: ¿algún lote fuera?

In [ ]:
fuera = x[(x > lsc_i) | (x < lic_i)]
print(lotes.loc[fuera.index, ["lote", "dia", "esteres"]])

**Celda 2.5** · Carta I-MR: la gráfica

In [ ]:
plt.plot(x.index + 1, x, marker="o")
for limite in (lic_i, media_x, lsc_i):
    plt.axhline(limite, linestyle="--")
plt.show()

**Celda 2.5b** · Carta I-MR: la gráfica del rango móvil

In [ ]:
plt.plot(rm.index + 1, rm, marker="o")
plt.axhline(rm_prom, linestyle="--")
plt.axhline(lsc_rm, linestyle="--")
plt.show()

**Celda 2.6** · Carta X̄-R: media y rango de cada día

In [ ]:
por_dia = lotes.groupby("dia")["esteres"]
xbarra = por_dia.mean()
rango = por_dia.max() - por_dia.min()
print(xbarra.head(3).round(3).tolist())
print(rango.head(3).round(2).tolist())

**Celda 2.7** · Carta X̄-R: gran media y rango promedio

In [ ]:
gran_media = xbarra.mean()
rango_prom = rango.mean()
print("gran media =", round(gran_media, 3))
print("rango promedio =", round(rango_prom, 3))

**Celda 2.8** · Carta X̄-R: los límites con A2, D3 y D4

In [ ]:
A2, D3, D4 = 0.577, 0, 2.114   # constantes para n = 5
lsc_x = gran_media + A2 * rango_prom
lic_x = gran_media - A2 * rango_prom
lsc_r = D4 * rango_prom
print("X̄: LIC =", round(lic_x, 3), "  LSC =", round(lsc_x, 3))
print("R:  LSC =", round(lsc_r, 3))

**Celda 2.9** · Carta X̄-R: ¿qué días quedan fuera?

In [ ]:
print("X̄ fuera:", xbarra[(xbarra > lsc_x) | (xbarra < lic_x)].round(3).to_dict())
print("R fuera:", rango[rango > lsc_r].round(2).to_dict())

**Celda 2.10** · Carta X̄-R: la gráfica

In [ ]:
fig, (arriba, abajo) = plt.subplots(2, 1, sharex=True)
arriba.plot(xbarra, marker="o")
abajo.plot(rango, marker="o")
for limite in (lic_x, gran_media, lsc_x):
    arriba.axhline(limite, linestyle="--")
abajo.axhline(lsc_r, linestyle="--")
plt.show()

**Celda 2.11** · El atajo que engaña: la desviación de todos los datos

In [ ]:
de_total = esteres.std()
ancho_bien = A2 * rango_prom
ancho_mal = 3 * de_total / 5 ** 0.5
print("con el rango promedio: ±", round(ancho_bien, 3))
print("con la DE de todo:     ±", round(ancho_mal, 3))

**Celda 2.12** · Carta X̄-S: la desviación de cada día

In [ ]:
s = por_dia.std()
s_prom = s.mean()
A3, B3, B4 = 1.427, 0, 2.089   # constantes para n = 5
lic_s, lsc_s = gran_media - A3 * s_prom, gran_media + A3 * s_prom
print("X̄: LIC =", round(lic_s, 3), "  LSC =", round(lsc_s, 3))
print("S:  LSC =", round(B4 * s_prom, 3))

**Celda 2.13** · Práctica: la carta I-MR de los lotes 41 a 80

In [ ]:
x2 = esteres.iloc[40:80]
rm2 = x2.diff().abs()
lsc2 = x2.mean() + 2.66 * rm2.mean()
lic2 = x2.mean() - 2.66 * rm2.mean()
print("LIC =", round(lic2, 3), "  LSC =", round(lsc2, 3))
fuera2 = x2[(x2 > lsc2) | (x2 < lic2)]
print("lotes fuera:", len(fuera2))

## Sección 3 · Cómo leer una carta de control

**Celda 3.1** · Las zonas: una sigma de la carta X̄

In [ ]:
sigma_x = A2 * rango_prom / 3      # una sigma de la carta X̄
zonas = [gran_media + k * sigma_x for k in (-3, -2, -1, 0, 1, 2, 3)]
print([round(float(z), 3) for z in zonas])

**Celda 3.2** · Regla 1: un punto más allá de tres sigmas

In [ ]:
def regla_1(medias, centro, sigma):
    return [int(d) for d, v in medias.items() if abs(v - centro) > 3 * sigma]
print(regla_1(xbarra, gran_media, sigma_x))

**Celda 3.3** · Regla 2: nueve puntos seguidos de un mismo lado

In [ ]:
def regla_2(medias, centro, n=9):
    racha, previo, alarmas = 0, 0, []
    for d, v in medias.items():
        lado = 1 if v > centro else -1
        racha = racha + 1 if lado == previo else 1
        previo = lado
        if racha >= n: alarmas.append(int(d))
    return alarmas

**Celda 3.4** · Regla 2 en la fase I

In [ ]:
print(regla_2(xbarra, gran_media))

**Celda 3.5** · Regla 3: seis puntos seguidos subiendo o bajando

In [ ]:
def regla_3(medias):
    v = list(medias.values)
    alarmas = []
    for i in range(5, len(v)):
        tramo = np.diff(v[i - 5:i + 1])
        if (tramo > 0).all() or (tramo < 0).all(): alarmas.append(int(medias.index[i]))
    return alarmas
print(regla_3(xbarra))

**Celda 3.6** · Regla 5: dos de tres más allá de dos sigmas

In [ ]:
def regla_5(medias, centro, sigma):
    z = list((medias - centro) / sigma)
    alarmas = []
    for i in range(2, len(z)):
        arriba, abajo = sum(t > 2 for t in z[i-2:i+1]), sum(t < -2 for t in z[i-2:i+1])
        if arriba >= 2 or abajo >= 2: alarmas.append(int(medias.index[i]))
    return alarmas
print(regla_5(xbarra, gran_media, sigma_x))

**Celda 3.7** · Falsas alarmas: 1,000 cartas de un proceso estable

In [ ]:
rng = np.random.default_rng(1)
solo_1, cuatro = 0, 0
for _ in range(1000):
    sim = pd.Series(rng.normal(0, 1, 25), index=range(1, 26))
    r1 = regla_1(sim, 0, 1)
    solo_1 += len(r1) > 0
    cuatro += len(r1 + regla_2(sim, 0) + regla_3(sim) + regla_5(sim, 0, 1)) > 0
print("con la regla 1:", solo_1, "de 1000")
print("con cuatro reglas:", cuatro, "de 1000")

**Celda 3.8** · Fase I: fuera los días con causa conocida

In [ ]:
limpios = lotes[~lotes["dia"].isin([7, 13, 14])]
g1 = limpios.groupby("dia")["esteres"]
centro = g1.mean().mean()
rbar1 = (g1.max() - g1.min()).mean()
lsc_f1, lic_f1 = centro + A2 * rbar1, centro - A2 * rbar1
print("centro =", round(centro, 3), "  R̄ =", round(rbar1, 3))
print("LIC =", round(lic_f1, 3), "  LSC =", round(lsc_f1, 3))

**Celda 3.9** · Fase II: los días nuevos contra límites fijos

In [ ]:
fase2 = pd.read_csv(URL + "lotes_fase2.csv")
xbarra2 = fase2.groupby("dia")["esteres"].mean()
sigma_f1 = A2 * rbar1 / 3
print("regla 1:", regla_1(xbarra2, centro, sigma_f1))
print("regla 2:", regla_2(xbarra2, centro))
print("regla 5:", regla_5(xbarra2, centro, sigma_f1))

**Celda 3.10** · Práctica: ¿y con ocho seguidos?

In [ ]:
print("ocho seguidos:", regla_2(xbarra2, centro, n=8))

## Sección 4 · Cartas de control para atributos

**Celda 4.1** · Carta p: la proporción de cada turno

In [ ]:
envases = pd.read_csv(URL + "envasado.csv")
envases["p"] = envases["rechazados"] / envases["inspeccionados"]
p_barra = envases["rechazados"].sum() / envases["inspeccionados"].sum()
print(envases.head(3).round(4))
print("p̄ =", round(p_barra, 4))

**Celda 4.2** · Carta p: límites que cambian con n

In [ ]:
error = (p_barra * (1 - p_barra) / envases["inspeccionados"]) ** 0.5
envases["lsc"] = p_barra + 3 * error
envases["lic"] = (p_barra - 3 * error).clip(lower=0)
print(envases[["turno", "inspeccionados", "lic", "lsc"]].head(3).round(4))

**Celda 4.3** · Carta p: ¿qué turno se sale?

In [ ]:
fuera_p = envases[(envases["p"] > envases["lsc"]) | (envases["p"] < envases["lic"])]
print(fuera_p[["turno", "inspeccionados", "rechazados", "p"]].round(3))

**Celda 4.3b** · Carta p: la gráfica, con límites escalonados

In [ ]:
plt.plot(envases["turno"], envases["p"], marker="o")
plt.step(envases["turno"], envases["lsc"], where="mid", linestyle="--")
plt.step(envases["turno"], envases["lic"], where="mid", linestyle="--")
plt.axhline(p_barra)
plt.show()

**Celda 4.4** · Carta u: defectos de etiquetado por bidón

In [ ]:
etiq = pd.read_csv(URL + "defectos.csv")
etiq["u"] = etiq["defectos"] / etiq["bidones"]
u_barra = etiq["defectos"].sum() / etiq["bidones"].sum()
lsc_u = u_barra + 3 * (u_barra / etiq["bidones"]) ** 0.5
print("ū =", round(u_barra, 4))
print(etiq.loc[etiq["u"] > lsc_u, ["dia", "bidones", "defectos", "u"]].round(3))

**Celda 4.4b** · Carta u: la gráfica

In [ ]:
plt.plot(etiq["dia"], etiq["u"], marker="o")
plt.step(etiq["dia"], lsc_u, where="mid", linestyle="--")
plt.axhline(u_barra)
plt.show()

**Celda 4.5** · Práctica: carta c de quejas por día

In [ ]:
quejas = pd.read_csv(URL + "quejas.csv")
c_barra = quejas["quejas"].mean()
lsc_c = c_barra + 3 * c_barra ** 0.5
lic_c = max(0, c_barra - 3 * c_barra ** 0.5)
print("c̄ =", round(c_barra, 2), "  LIC =", round(lic_c, 2), "  LSC =", round(lsc_c, 2))
print("días fuera:", quejas.loc[quejas["quejas"] > lsc_c, "dia"].tolist())

## Sección 5 · Capacidad del proceso

**Celda 5.1** · Llenado: la sigma de corto plazo

In [ ]:
llenado = pd.read_csv(URL + "llenado.csv")
por_hora = llenado.groupby("hora")["volumen"]
rango_ll = (por_hora.max() - por_hora.min()).mean()
sigma_ll = rango_ll / 2.059          # d2 para n = 4
media_ll = llenado["volumen"].mean()
print("media =", round(media_ll, 4), "  sigma =", round(sigma_ll, 4))

**Celda 5.2** · Cp y Cpk del llenado

In [ ]:
LIE, LSE = 19.85, 20.15
cp = (LSE - LIE) / (6 * sigma_ll)
cpk = min(LSE - media_ll, media_ll - LIE) / (3 * sigma_ll)
print("Cp  =", round(cp, 3))
print("Cpk =", round(cpk, 3))

**Celda 5.3** · Cpk de los ésteres: un solo lado

In [ ]:
sigma_e = rbar1 / 2.326              # d2 para n = 5, fase I depurada
cpk_e = (centro - LEI) / (3 * sigma_e)
print("sigma =", round(sigma_e, 4))
print("Cpk (solo inferior) =", round(cpk_e, 3))

**Celda 5.4** · Ppk: con la desviación de todos los datos

In [ ]:
de_largo = limpios["esteres"].std()
ppk_e = (limpios["esteres"].mean() - LEI) / (3 * de_largo)
print("DE de largo plazo =", round(de_largo, 4))
print("Ppk =", round(ppk_e, 3))

**Celda 5.5** · Normalidad: Shapiro-Wilk y gráfico Q-Q

In [ ]:
from scipy import stats
w, p_sw = stats.shapiro(limpios["esteres"])
print("W =", round(w, 3), "  p =", round(p_sw, 3))
stats.probplot(limpios["esteres"], plot=plt)
plt.show()

**Celda 5.6** · De Cpk a partes por millón

In [ ]:
z = 3 * cpk_e
ppm = stats.norm.cdf(-z) * 1_000_000
print("z =", round(z, 2))
print("partes por millón bajo 96.5 % =", round(ppm))
print("observados:", (limpios["esteres"] < LEI).sum(), "de", len(limpios))

**Celda 5.7** · El intervalo de confianza de Cpk

In [ ]:
n_e = len(limpios)
ee = (1 / (9 * n_e) + cpk_e ** 2 / (2 * (n_e - 1))) ** 0.5
print("Cpk =", round(cpk_e, 3))
print("IC 95 %:", round(cpk_e - 1.96 * ee, 3), "a", round(cpk_e + 1.96 * ee, 3))

**Celda 5.8** · ¿Cuántos datos bastan?

In [ ]:
for n_i in (30, 50, 100, 200):
    ee_i = (1 / (9 * n_i) + cpk_e ** 2 / (2 * (n_i - 1))) ** 0.5
    print(n_i, "datos: ±", round(1.96 * ee_i, 3))

**Celda 5.9** · Práctica: si la llenadora se centra en 20.00 L

In [ ]:
cpk_centrado = min(LSE - 20.00, 20.00 - LIE) / (3 * sigma_ll)
print("Cpk centrado:", round(cpk_centrado, 3), "  Cp:", round(cp, 3))

## Sección 6 · Experimentar en lugar de adivinar

**Celda 6.1** · Tu hoja de corridas aleatorizada

In [ ]:
rng = np.random.default_rng(2026)
niveles = [0.5, 0.75, 1.0]          # % de catalizador
hoja = pd.DataFrame({"catalizador": np.repeat(niveles, 4)})
hoja["orden"] = rng.permutation(len(hoja)) + 1
print(hoja.sort_values("orden").head(6))

**Celda 6.2** · Los datos del experimento

In [ ]:
exp1 = pd.read_csv(URL + "experimento_1factor.csv")
medias_c = exp1.groupby("catalizador")["rendimiento"].mean()
media_global = exp1["rendimiento"].mean()
print(medias_c.round(2))
print("media global =", round(media_global, 3))

**Celda 6.3** · Suma de cuadrados entre tratamientos, a mano

In [ ]:
n_c = exp1.groupby("catalizador").size()
sc_trat = (n_c * (medias_c - media_global) ** 2).sum()
print("SC tratamientos =", round(sc_trat, 4))

**Celda 6.4** · Suma de cuadrados del error, a mano

In [ ]:
media_de_su_grupo = exp1.groupby("catalizador")["rendimiento"].transform("mean")
sc_error = ((exp1["rendimiento"] - media_de_su_grupo) ** 2).sum()
print("SC error =", round(sc_error, 4))

**Celda 6.5** · La F, a mano

In [ ]:
from scipy import stats
gl_trat, gl_error = 3 - 1, len(exp1) - 3
cm_trat, cm_error = sc_trat / gl_trat, sc_error / gl_error
F = cm_trat / cm_error
p_valor = stats.f.sf(F, gl_trat, gl_error)
print("CM trat =", round(cm_trat, 4), "  CM error =", round(cm_error, 4))
print("F =", round(F, 2), "  p =", round(p_valor, 4))

**Celda 6.6** · Comprueba con statsmodels

In [ ]:
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm
modelo1 = smf.ols("rendimiento ~ C(catalizador)", data=exp1).fit()
print(anova_lm(modelo1).round(4))

**Celda 6.7** · Bloques: con y sin el lote de aceite

In [ ]:
bloq = pd.read_csv(URL + "bloques.csv")
sin = anova_lm(smf.ols("rendimiento ~ C(catalizador)", data=bloq).fit())
modelo_b = smf.ols("rendimiento ~ C(catalizador) + C(lote_aceite)", data=bloq).fit()
con = anova_lm(modelo_b)
for nombre, tabla in (("sin bloques", sin), ("con bloques", con)):
    f_cat, p_cat = tabla.loc["C(catalizador)", ["F", "PR(>F)"]]
    print(nombre, ": F =", round(f_cat, 2), "  p =", round(p_cat, 4))

**Celda 6.8** · Práctica: ¿cuánto pesa el lote de aceite?

In [ ]:
print(con.round(4))
print(bloq.groupby("lote_aceite")["rendimiento"].mean().round(2))

## Sección 7 · Diseños factoriales 2^k

**Celda 7.1** · Codifica los factores en −1 y +1

In [ ]:
fac = pd.read_csv(URL + "factorial_2k.csv")
altos = {"temperatura": 60, "catalizador": 1.0, "relacion": 7}
for col, alto in altos.items():
    fac[col[0].upper()] = np.where(fac[col] == alto, 1, -1)
print(fac[["temperatura", "catalizador", "relacion", "T", "C", "R"]].head(4))

**Celda 7.2** · La tabla de signos

In [ ]:
fac["TC"] = fac["T"] * fac["C"]
fac["TR"] = fac["T"] * fac["R"]
fac["CR"] = fac["C"] * fac["R"]
fac["TCR"] = fac["T"] * fac["C"] * fac["R"]
orden_std = fac.sort_values(["replica", "corrida_std"])
print(orden_std[["T", "C", "R", "TC", "rendimiento"]].head(8))

**Celda 7.3** · Los efectos: media en + menos media en −

In [ ]:
efectos = {}
for col in ["T", "C", "R", "TC", "TR", "CR", "TCR"]:
    en_alto = fac.loc[fac[col] == 1, "rendimiento"].mean()
    en_bajo = fac.loc[fac[col] == -1, "rendimiento"].mean()
    efectos[col] = en_alto - en_bajo
print(pd.Series(efectos).round(3))

**Celda 7.4** · El modelo con statsmodels

In [ ]:
modelo3 = smf.ols("rendimiento ~ T * C * R", data=fac).fit()
print(modelo3.params.round(3))
print("2 × coeficiente de T =", round(2 * modelo3.params["T"], 3))

**Celda 7.5** · El ANOVA del factorial

In [ ]:
print(anova_lm(modelo3, typ=2).round(4))

**Celda 7.6** · Revisa los residuos

In [ ]:
residuos = modelo3.resid
w_r, p_r = stats.shapiro(residuos)
print("Shapiro-Wilk de los residuos: W =", round(w_r, 3), "  p =", round(p_r, 3))
fig, (izq, der) = plt.subplots(1, 2)
stats.probplot(residuos, plot=izq)
der.scatter(modelo3.fittedvalues, residuos)
plt.show()

**Celda 7.7** · La gráfica de interacción

In [ ]:
tabla_tc = fac.groupby(["temperatura", "catalizador"])["rendimiento"].mean()
print(tabla_tc.round(2))
tabla_tc.unstack().plot(marker="o")
plt.ylabel("rendimiento (%)")
plt.show()

**Celda 7.8** · El modelo reducido

In [ ]:
reducido = smf.ols("rendimiento ~ T + C + T:C", data=fac).fit()
print(reducido.params.round(3))
print("R² ajustado =", round(reducido.rsquared_adj, 3))

**Celda 7.9** · Práctica: la mejor esquina

In [ ]:
esquinas = pd.DataFrame({"T": [-1, 1, -1, 1], "C": [-1, -1, 1, 1]})
esquinas["predicho"] = reducido.predict(esquinas).round(2)
print(esquinas)

## Sección 8 · Cribado: factoriales sin réplicas y fraccionados

**Celda 8.1** · Las 16 corridas sin réplicas

In [ ]:
cri = pd.read_csv(URL + "cribado_2_5-1.csv")
fact = cri[cri["corrida_std"] <= 16].copy()
centros = cri[cri["corrida_std"] > 16]
nombres = {"A": "temperatura", "B": "relacion", "C": "catalizador",
           "D": "tiempo", "E": "agitacion"}
for letra, col in nombres.items():
    fact[letra] = np.where(fact[col] == fact[col].max(), 1, -1)
print(fact.sort_values("corrida_std")[["A", "B", "C", "D", "rendimiento"]].head(4))

**Celda 8.2** · Los 15 efectos de un 2⁴

In [ ]:
from itertools import combinations
def efecto(datos, combo):
    signo = datos[list(combo)].prod(axis=1)
    alto = datos.loc[signo == 1, "rendimiento"].mean()
    return alto - datos.loc[signo == -1, "rendimiento"].mean()
combos = [c for r in (1, 2, 3, 4) for c in combinations("ABCD", r)]
efectos8 = {"".join(c): efecto(fact, c) for c in combos}
print(pd.Series(efectos8).round(2).sort_values())

**Celda 8.3** · La gráfica normal de efectos

In [ ]:
ordenados = pd.Series(efectos8).sort_values()
cuantiles = stats.norm.ppf((np.arange(1, 16) - 0.5) / 15)
plt.scatter(cuantiles, ordenados.values)
for q, (nombre, v) in zip(cuantiles, ordenados.items()):
    plt.annotate(nombre, (q, v))
plt.show()

**Celda 8.4** · El método de Lenth

In [ ]:
absolutos = np.abs(np.array(list(efectos8.values())))
s0 = 1.5 * np.median(absolutos)
pse = 1.5 * np.median(absolutos[absolutos < 2.5 * s0])
margen = stats.t.ppf(0.975, len(absolutos) / 3) * pse
print("s0 =", round(s0, 3), "  PSE =", round(pse, 3), "  margen =", round(margen, 3))
print("activos:", [k for k, v in efectos8.items() if abs(v) > margen])

**Celda 8.5** · Construye el 2^(5−1) con pyDOE3

In [ ]:
from pyDOE3 import fracfact
diseno = fracfact("a b c d abcd")
diseno = diseno[:, [3, 2, 1, 0, 4]]    # orden estándar: A cambia más rápido
tabla = pd.DataFrame(diseno, columns=list("ABCDE")).astype(int)
print(tabla.head(4))
print("¿E = ABCD?", bool((tabla["E"] == tabla[list("ABCD")].prod(axis=1)).all()))

**Celda 8.6** · Los alias con pyDOE3

In [ ]:
from pyDOE3 import fracfact_aliasing
alias, _ = fracfact_aliasing(fracfact("a b c d abcd"))
print(alias[:6])
print([a for a in alias if a.startswith("ac ")])

**Celda 8.7** · Analiza el 2^(5−1) de la planta

In [ ]:
pares = list(combinations("ABCDE", 1)) + list(combinations("ABCDE", 2))
efectos16 = {"".join(c): efecto(fact, c) for c in pares}
abs16 = np.abs(np.array(list(efectos16.values())))
pse16 = 1.5 * np.median(abs16[abs16 < 2.5 * 1.5 * np.median(abs16)])
margen16 = stats.t.ppf(0.975, 15 / 3) * pse16
print("activos:", {k: round(float(v), 2) for k, v in efectos16.items() if abs(v) > margen16})

**Celda 8.8** · Puntos centrales: ¿hay curvatura?

In [ ]:
media_fact = fact["rendimiento"].mean()
media_centro = centros["rendimiento"].mean()
s_c = centros["rendimiento"].std()
t_c = (media_fact - media_centro) / (s_c * (1 / 16 + 1 / 4) ** 0.5)
p_c = 2 * stats.t.sf(abs(t_c), df=3)
print("esquinas =", round(media_fact, 2), "  centros =", round(media_centro, 2))
print("t =", round(t_c, 2), "  p", "< 0.001" if p_c < 0.001 else f"= {p_c:.4f}")

**Celda 8.9** · Práctica: ¿con qué se confunde la interacción BC?

In [ ]:
print([a for a in alias if a.startswith("bc ")])
print("efecto BC =", round(efectos16["BC"], 2), "  margen =", round(margen16, 2))

## Sección 9 · Superficie de respuesta: encuentra el óptimo

**Celda 9.1** · Construye el central compuesto

In [ ]:
from pyDOE3 import ccdesign
cc = ccdesign(2, center=(0, 5), alpha="rotatable", face="ccc")
print(np.round(cc, 3))

**Celda 9.2** · Pásalo a unidades reales

In [ ]:
real = pd.DataFrame({"temperatura": 58 + 4 * cc[:, 0],
                     "catalizador": 0.85 + 0.2 * cc[:, 1]})
print(real.round(3))

**Celda 9.3** · Ajusta el modelo cuadrático

In [ ]:
ccd_d = pd.read_csv(URL + "ccd.csv")
cuad = smf.ols("rendimiento ~ x1 + x2 + I(x1**2) + I(x2**2) + x1:x2", data=ccd_d).fit()
print(cuad.params.round(3))
print("R² =", round(cuad.rsquared, 3), "  R² ajustado =", round(cuad.rsquared_adj, 3))

**Celda 9.4** · ¿Qué términos importan?

In [ ]:
print(cuad.pvalues.round(4))

**Celda 9.5** · La superficie y sus contornos

In [ ]:
x1g, x2g = np.meshgrid(np.linspace(-1.5, 1.5, 61), np.linspace(-1.5, 1.5, 61))
malla = pd.DataFrame({"x1": x1g.ravel(), "x2": x2g.ravel()})
z = cuad.predict(malla).values.reshape(x1g.shape)
curvas = plt.contour(58 + 4 * x1g, 0.85 + 0.2 * x2g, z, levels=10)
plt.clabel(curvas)
plt.xlabel("temperatura (°C)")
plt.ylabel("catalizador (%)")
plt.show()

**Celda 9.6** · El punto estacionario

In [ ]:
b = cuad.params
B = np.array([[b["I(x1 ** 2)"], b["x1:x2"] / 2], [b["x1:x2"] / 2, b["I(x2 ** 2)"]]])
xs = -0.5 * np.linalg.solve(B, [b["x1"], b["x2"]])
print("codificado:", xs.round(3))
T_opt, C_opt = 58 + 4 * xs[0], 0.85 + 0.2 * xs[1]
print("real: T =", round(T_opt, 1), "°C   catalizador =", round(C_opt, 3), "%")
print("autovalores:", np.linalg.eigvalsh(B).round(3))

**Celda 9.7** · La predicción en el óptimo

In [ ]:
en_opt = pd.DataFrame({"x1": [xs[0]], "x2": [xs[1]]})
pred = cuad.get_prediction(en_opt).summary_frame(alpha=0.05)
print(pred[["mean", "obs_ci_lower", "obs_ci_upper"]].round(2))

**Celda 9.8** · La función de deseabilidad

In [ ]:
def d_mayor(y, bajo, alto):      # más es mejor
    return np.clip((y - bajo) / (alto - bajo), 0, 1)
def d_menor(y, bajo, alto):      # menos es mejor
    return np.clip((alto - y) / (alto - bajo), 0, 1)
T_m, C_m = 58 + 4 * x1g, 0.85 + 0.2 * x2g
costo_m = 300 * C_m + 1.5 * (T_m - 50)      # MXN por tonelada de aceite
D = (d_mayor(z, 95, 97.2) * d_menor(costo_m, 200, 350)) ** 0.5

**Celda 9.9** · El mejor compromiso

In [ ]:
i = np.unravel_index(D.argmax(), D.shape)
print("T =", round(T_m[i], 1), "°C   catalizador =", round(C_m[i], 3), "%")
print("rendimiento =", round(z[i], 2), "%")
print("costo =", round(costo_m[i], 1), "MXN/t   D =", round(D[i], 3))

**Celda 9.10** · Práctica: ¿y si el costo no importara?

In [ ]:
j = np.unravel_index(z.argmax(), z.shape)
print("máximo rendimiento:", round(z[j], 2), "%")
print("en T =", round(T_m[j], 1), "°C y catalizador =", round(C_m[j], 3), "%")
print("costo ahí =", round(costo_m[j], 1), "MXN/t")

## Sección 10 · ¿Puedes confiar en tu medición? Gauge R&R

**Celda 10.1** · El ANOVA del estudio

In [ ]:
gr = pd.read_csv(URL + "gauge_rr.csv")
modelo_g = smf.ols("esteres ~ C(lote) * C(analista)", data=gr).fit()
tabla_g = anova_lm(modelo_g, typ=1)
print(tabla_g[["df", "sum_sq", "mean_sq"]].round(5))

**Celda 10.2** · De los cuadrados medios a las varianzas

In [ ]:
cm = tabla_g["mean_sq"]
a, b, r = 10, 3, 2      # lotes, analistas, réplicas
var_rep = cm["Residual"]
var_inter = max(0, (cm["C(lote):C(analista)"] - cm["Residual"]) / r)
var_anal = max(0, (cm["C(analista)"] - cm["C(lote):C(analista)"]) / (a * r))
var_lote = max(0, (cm["C(lote)"] - cm["C(lote):C(analista)"]) / (b * r))
print("repetibilidad:", round(var_rep, 5), "  analista:", round(var_anal, 5))
print("interacción:", round(var_inter, 5), "  lote:", round(var_lote, 5))

**Celda 10.3** · %GRR y categorías distintas

In [ ]:
var_grr = var_rep + var_anal + var_inter
var_total = var_grr + var_lote
pct_grr = 100 * (var_grr / var_total) ** 0.5
ndc = int(1.41 * (var_lote / var_grr) ** 0.5)
print("%GRR =", round(pct_grr, 1), "%")
print("categorías distintas =", ndc)

**Celda 10.4** · Las gráficas del estudio

In [ ]:
fig, (izq, der) = plt.subplots(1, 2)
gr.boxplot(column="esteres", by="analista", ax=izq)
gr.groupby(["lote", "analista"])["esteres"].mean().unstack().plot(ax=der, marker="o")
plt.show()
print(gr.groupby("analista")["esteres"].mean().round(3))

**Celda 10.5** · Práctica: si los analistas midieran igual

In [ ]:
pct_solo_rep = 100 * (var_rep / (var_rep + var_lote)) ** 0.5
print("%GRR solo con la repetibilidad:", round(pct_solo_rep, 1), "%")

## Sección 11 · Proyecto integrador y proyecto final

**Celda 11.1** · Corridas de confirmación

In [ ]:
conf = pd.read_csv(URL + "confirmacion.csv")
print("observados:", conf["rendimiento"].tolist())
print("media =", round(conf["rendimiento"].mean(), 2))
pi_inf, pi_sup = pred["obs_ci_lower"].iloc[0], pred["obs_ci_upper"].iloc[0]
print("intervalo de predicción: de", round(pi_inf, 2), "a", round(pi_sup, 2))

**Celda 11.2** · Nuevos límites para el proceso mejorado

In [ ]:
fase3 = pd.read_csv(URL + "lotes_fase3.csv")
g3 = fase3.groupby("dia")["esteres"]
centro3 = g3.mean().mean()
rbar3 = (g3.max() - g3.min()).mean()
print("centro =", round(centro3, 3), "  R̄ =", round(rbar3, 3))
lic3, lsc3 = centro3 - A2 * rbar3, centro3 + A2 * rbar3
print("X̄: LIC =", round(lic3, 3), "  LSC =", round(lsc3, 3))
print("R:  LSC =", round(D4 * rbar3, 3))

**Celda 11.3** · La capacidad antes y después

In [ ]:
cpk_despues = (centro3 - LEI) / (3 * rbar3 / 2.326)
print("Cpk antes   =", round(cpk_e, 3))
print("Cpk después =", round(cpk_despues, 3))
print("ppm antes   =", round(stats.norm.cdf(-3 * cpk_e) * 1e6))
print("ppm después =", round(stats.norm.cdf(-3 * cpk_despues) * 1e6, 2))

**Celda 11.4** · Proyecto: la carta X̄-R de la línea 2

In [ ]:
pl = pd.read_csv(URL + "llenado_proyecto.csv")
gp = pl.groupby("hora")["volumen"]
xb_p, r_p = gp.mean(), gp.max() - gp.min()
A2_4, D4_4 = 0.729, 2.282          # constantes para n = 4
lsc_p, lic_p = xb_p.mean() + A2_4 * r_p.mean(), xb_p.mean() - A2_4 * r_p.mean()
print("X̄ fuera:", xb_p[(xb_p > lsc_p) | (xb_p < lic_p)].round(3).to_dict())
print("R fuera:", r_p[r_p > D4_4 * r_p.mean()].round(3).to_dict())

**Celda 11.5** · Proyecto: la capacidad sin la hora 12

In [ ]:
sin12 = pl[pl["hora"] != 12]
g12 = sin12.groupby("hora")["volumen"]
sigma_p = (g12.max() - g12.min()).mean() / 2.059
media_p = sin12["volumen"].mean()
print("sigma =", round(sigma_p, 4), "  Cp =", round(0.30 / (6 * sigma_p), 3))
print("Cpk =", round(min(LSE - media_p, media_p - LIE) / (3 * sigma_p), 3))

**Celda 11.6** · Proyecto: el experimento 2²

In [ ]:
ep = pd.read_csv(URL + "experimento_proyecto.csv")
print(ep.groupby(["velocidad", "boquilla"])["de_ml"].mean().round(1))
mod_p = smf.ols("de_ml ~ C(velocidad) * C(boquilla)", data=ep).fit()
print(anova_lm(mod_p, typ=2).round(4))

**Celda 11.7** · Proyecto: la decisión

In [ ]:
mejor = ep.query("velocidad == 'lenta' and boquilla == 'nueva'")["de_ml"].mean()
print("DE con velocidad lenta y boquilla nueva =", round(mejor, 1), "mL")
print("Cp esperado =", round(300 / (6 * mejor), 2))